In [1]:
import requests as r
import json
import pandas as pd


In [31]:
url_skill = 'https://raw.githubusercontent.com/LartTyler/mhdb-wilds-data/refs/heads/main/output/merged/Skill.json'
response_skill = r.get(url_skill)
skill_request =json.loads(response_skill.text)
df_skills = pd.json_normalize(skill_request)

df_skills['descriptions.en'] = df_skills['descriptions.en'].str.replace('\r\n',' ')
df_skills['slug'] = df_skills['names.en'].str.lower().str.replace(' ','_')
df_skills['max_level'] = df_skills['ranks'].apply(lambda x: len(x) if isinstance(x, list) else 0)
skills_features = ['slug','kind','icon','max_level','game_id']
df_skills = df_skills[skills_features]
df_skills.index.name = 'id' 
#creating a map dateframe to convert id of other dateframes
df_map = df_skills[['slug','game_id']]
df_map.reset_index(inplace=True)
df_map.head(5)


,id,slug,game_id
0,0,dragon_resistance,-2125233152
1,1,spread/power_shots,-2123993856
2,2,critical_eye,-2096489472
3,3,guardian's_protection,-2041451904
4,4,blast_functionality,-2022542848


In [ ]:
#creating skills dateframe
df_skills.drop(columns=['game_id'],inplace=True)
df_skills.head(5)

,slug,kind,icon,max_level
id,,,,
0,dragon_resistance,armor,defense,3
1,spread/power_shots,weapon,ranged,1
2,critical_eye,weapon,affinity,5
3,guardian's_protection,group,group,1
4,blast_functionality,weapon,ranged,1


In [43]:
df_skill_trans = pd.json_normalize(skill_request)
name_traduction_list = [traduction for traduction in df_skill_trans.columns.tolist() if traduction.startswith('names.')]
desc_traduction_list = [traduction for traduction in df_skill_trans.columns.tolist() if traduction.startswith('descriptions.')]
df_skills_name_trans = pd.melt(df_skill_trans,id_vars='game_id',value_vars= name_traduction_list,var_name = 'language',value_name='name')
df_skills_name_trans['language'] = df_skills_name_trans['language'].str.replace('names.','')

df_skills_desc_trans = pd.melt(df_skill_trans,id_vars= "game_id",value_vars= desc_traduction_list, var_name='language',value_name='description')
df_skills_desc_trans['language'] = df_skills_desc_trans['language'].str.replace('descriptions.','')
df_skills_desc_trans['description']=df_skills_desc_trans['description'].str.replace('\r\n',' ')
df_skill_trans = pd.merge(df_skills_name_trans,df_skills_desc_trans,how='inner',on=['game_id','language'])
df_skill_trans = pd.merge(df_skill_trans,df_map,how='left',on='game_id')
df_skill_trans = df_skill_trans[['id','slug','language','name','description']].rename(columns= {'id':'skill_id'})
df_skill_trans.index.name='id'
df_skill_trans.sample(5)


,skill_id,slug,language,name,description
id,,,,,
2560,54,stench_resistance,es-419,Antihedor,Otorga protección contra el hedor.
2533,27,leap_of_faith,es-419,Mejora de salto,Te permite evadir lanzándote cuando estás de c...
794,78,synthetic_shield,de,Synthetischer Schild,Verbessert vorübergehend deine Verteidigung un...
954,59,focus,es,Carga veloz,"Carga los indicadores de armas rápidamente, as..."
2421,94,seregios's_tenacity,th,เซเรกิออสใจสู้,NaN


In [54]:
language_map = {
    'it': 'Italiano',
    'en': 'English',
    'fr': 'Français',
    'de': 'Deutsch',
    'es': 'Español (España)',
    'es-419': 'Español (Latinoamérica)',
    'pt-BR': 'Português (Brasil)',
    'zh-Hans': '简体中文',
    'zh-Hant': '繁體中文',
    'ja': '日本語',
    'ko': '한국어',
    'ru': 'Русский',
    'pl': 'Polski',
    'ar': 'العربية',
    'th': 'ไทย'
}
df_skill_trans['language'] = df_skill_trans['language'].replace(language_map)
df_skill_trans['language'] = df_skill_trans['language'].replace('Español (España)-419','Español (Latinoamérica)')
df_skill_trans.sample(15)

,skill_id,slug,language,name,description
id,,,,,
2353,26,guardian's_pulse,ไทย,ชีพจรการ์เดียน,NaN
1553,121,maximum_might,Português (Brasil),Poder Máximo,Aumenta a afinidade ao manter o vigor cheio po...
1916,126,power_prolonger,繁體中文,強化持續,延長太刀、雙劍、斬擊斧、充能斧和 操蟲棍的強化狀態時間。
1145,71,attack_boost,Русский,Усиление атаки,Повышает силу атаки.
1623,12,coalescence,한국어,전화위복,"상태 이상, 속성 피해를 해제하면 일정 시간 속성치, 상태 이상치가 상승한다."
1156,82,dreamspell_prayer,Русский,Молитва Чар снов,NaN
157,157,iron_skin,日本語,防御力ＤＯＷＮ耐性,防御力DOWNに対する耐性を持つ。
77,77,gogmapocalypse,日本語,巨戟龍の黙示録,NaN
244,65,offensive_guard,English,Offensive Guard,Temporarily increases attack power after execu...
